In [10]:
import glob
import os
import h5py
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400 # User can set this outside the class if needed

from astropy.io import fits
from astropy.io.votable import parse_single_table
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.coordinates import SkyCoord
import astropy.units as u

import os
import h5py
from tqdm import tqdm
from multiprocessing import Pool 
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import numpy as np

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
import astropy.units as u
from astropy.table import Table

from reproject import reproject_interp
from reproject import reproject_exact
from reproject import reproject_adaptive

from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr
import sys 
import gc 
from functools import partial
from IPython.display import clear_output

from SelfCal.MapHelper import bit_to_bool, make_weight, find_outliers, map_pixels
from SelfCal.WCSHelper import load_from_fits, save_to_fits, find_optimal_frame
from SelfCal import MakeMap
from SelfCal.SPHERExUtility import make_fiducial_chunk_map, make_fiducial_chunk_mask, interpolate_array, load_calibration, interp_2d_vertical, interp_1d
from SelfCal.MakeMap import load_reproj_file
import importlib
importlib.reload(MakeMap)

from datetime import datetime
import pandas as pd


In [2]:
def plot_map(map, wcs=None, lowp=0.1, highp=95, fig=None, ax=None, colorbar=True, low=None, high=None, cmap='viridis', norm=LogNorm,
            cbar_label='MJy/sr'):
    if low is None or high is None:
        high, low = np.nanpercentile(map[map>0], [highp, lowp])
    if fig is None:
        fig = plt.figure(figsize=(10, 10))
    if ax is None:
        ax = fig.add_subplot(111, projection=wcs)
    if norm is LogNorm:
        im = ax.imshow(map, norm=LogNorm(vmin=low, vmax=high), origin='lower', cmap=cmap)
    else:
        im = ax.imshow(map, vmin=low, vmax=high, origin='lower', cmap=cmap)

    if wcs is not None:
        # Explicitly set axis labels
        ax.coords['ra'].set_axislabel('RA')
        ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
        ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
        ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
        ax.coords['dec'].set_axislabel('DEC')
        ax.grid(color='black', linestyle='--', alpha=0.5)

    if colorbar:
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
        cbar.set_label(cbar_label)
    return fig, ax

In [5]:
DETECTOR = 1
OVERSAMPLE_FACTOR = 2
NUM_SUBCHANNELS = 10
NUM_CHANNELS = 17
NUM_VERTICAL_BANDS = 5

det_BC, det_BW = load_calibration(band=DETECTOR, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')
grid_subchannel_map, lvf_params = make_fiducial_chunk_map(DETECTOR, det_BC, num_subchannels=NUM_SUBCHANNELS, num_channels=NUM_CHANNELS, 
                                                oversample_factor=OVERSAMPLE_FACTOR)
det_subchannel_map, _ = make_fiducial_chunk_map(DETECTOR, det_BC, num_subchannels=NUM_SUBCHANNELS, num_channels=NUM_CHANNELS, 
                                           oversample_factor=1, lvf_params=lvf_params)

Fitting LVF parameters...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 171/171 [00:04<00:00, 37.47it/s]


Making chunk map...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 171/171 [00:06<00:00, 27.22it/s]


Making chunk map...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 171/171 [00:01<00:00, 121.66it/s]


In [6]:
grid_vertchunk_map = np.zeros_like(grid_subchannel_map)
det_vertchunk_map = np.zeros_like(det_subchannel_map)

for band in range(NUM_VERTICAL_BANDS):
    grid_width = grid_vertchunk_map.shape[1] // NUM_VERTICAL_BANDS
    grid_vertchunk_map[:, band*grid_width:(band+1)*grid_width] = band
    det_width = det_vertchunk_map.shape[1] // NUM_VERTICAL_BANDS
    det_vertchunk_map[:, band*det_width:(band+1)*det_width] = band

grid_chunk_map = grid_subchannel_map*NUM_VERTICAL_BANDS + grid_vertchunk_map
det_chunk_map = det_subchannel_map*NUM_VERTICAL_BANDS + det_vertchunk_map

ch = [7]

subchannel_valid_mask = make_fiducial_chunk_mask(ch, num_subchannels=NUM_SUBCHANNELS, num_channels=NUM_CHANNELS, padding=1)

chunk_valid_mask = np.zeros(len(subchannel_valid_mask)*NUM_VERTICAL_BANDS, dtype=subchannel_valid_mask.dtype)
for band in range(NUM_VERTICAL_BANDS):
    chunk_valid_mask[band::NUM_VERTICAL_BANDS] = subchannel_valid_mask

grid_valid_mask = chunk_valid_mask[grid_chunk_map]
det_valid_mask = chunk_valid_mask[det_chunk_map]

In [ ]:
cal_file = '/mnt/md124/thomasli/selfcal/outputs/nep_det1_6p2arcsec/calibration/cal_D1_Ch7_vertical5bands.h5'
with h5py.File(cal_file, 'r') as f:
    raw_offset = f['O'][:]

In [7]:
reproj_list = sorted(glob.glob(f'/mnt/md124/thomasli/selfcal/outputs/nep_det{DETECTOR}_6p2arcsec/reprojected/*.h5'))

In [20]:
reproj_path = reproj_list[0]
reproj_data = load_reproj_file(reproj_path, fields=['file_path', 'sub_mapping'])

In [21]:
sub_mapping = reproj_data['sub_mapping']

In [ ]:
det_to_sub(det_data, sub_mapping=None, interp_matrix=None)

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]],
      shape=(2, 3153, 3153), dtype=float32)

In [4]:
raw_offset.shape

(4412, 860)